# GRU

## Load data

Set directory

In [1]:
import sys
import os

os.environ["WANDB_START_METHOD"] = "thread"
os.environ["WANDB__SERVICE_WAIT"] = "300"

# Find the project root (Speciale_Kode)
current_dir = os.getcwd()
project_root = current_dir

# Looks for "Speciale_Kode" folder:
while os.path.basename(project_root) != "Speciale_Kode":
    project_root = os.path.dirname(project_root)

# Add to Python path
if project_root not in sys.path:
    sys.path.append(project_root)

Uses full features from read_data to predict DKPrice with cross-validation.

In [2]:
import pandas as pd
from pathlib import Path
from Modules.read_data import read_data

PRICE_ZONE = "DK1"  # "DK1" or "DK2"
TRAIN_WINDOW = 2 * 8760
VAL_START = "2024-01-01 00:00:00"
VAL_WINDOW = 8760
PREDICT_PERIOD = 4 * 168
STRIDE = 9 * 168                # Stride starts from the end of the predict period.

(
    DK1_train,
    DK1_test,
    DK2_train,
    DK2_test,
    DK1_train_weather,
    DK1_test_weather,
    DK2_train_weather,
    DK2_test_weather
) = read_data("combined_data_cleaned_v5.csv")

if PRICE_ZONE == "DK1":
    dataset_train = DK1_train.copy()
    dataset_test = DK1_test.copy()
elif PRICE_ZONE == "DK2":
    dataset_train = DK2_train.copy()
    dataset_test = DK2_test.copy()
else:
    raise ValueError("PRICE_ZONE must be 'DK1' or 'DK2'.")

# read_data already returns all of 2024 in dataset_train and all of 2025 in dataset_test.
# Build the custom 2024 rolling validation split entirely from dataset_train.
dataset_train = dataset_train.sort_values("Time").reset_index(drop=True)
dataset_test = dataset_test.sort_values("Time").reset_index(drop=True)

val_start_ts = pd.Timestamp(VAL_START)
year_2024_start = pd.Timestamp("2024-01-01 00:00:00")
year_2025_start = pd.Timestamp("2025-01-01 00:00:00")

# Keep legacy full timeline variable before redefining dataset_train below.
df = pd.concat([dataset_train, dataset_test], ignore_index=True).sort_values("Time").reset_index(drop=True)

# Fixed history block: TRAIN_WINDOW ending at VAL_START.
history = dataset_train.loc[dataset_train["Time"] < val_start_ts].copy().iloc[-TRAIN_WINDOW:]
if len(history) < TRAIN_WINDOW:
    raise ValueError(
        f"Not enough history for TRAIN_WINDOW={TRAIN_WINDOW}. Got {len(history)} rows before {VAL_START}."
    )

data_2024 = dataset_train.loc[
    (dataset_train["Time"] >= year_2024_start) & (dataset_train["Time"] < year_2025_start)
] .copy()

validation_idx = []
validation_windows = []
window_start = val_start_ts

# Validation windows in 2024: PREDICT_PERIOD, then STRIDE gap, repeat.
# Only full windows are allowed; trailing partial windows are skipped.
while (window_start + pd.Timedelta(hours=PREDICT_PERIOD)) <= year_2025_start:
    window_end = window_start + pd.Timedelta(hours=PREDICT_PERIOD)
    if window_end <= window_start:
        break

    mask = (data_2024["Time"] >= window_start) & (data_2024["Time"] < window_end)
    if mask.any():
        validation_idx.extend(data_2024.index[mask].tolist())
        validation_windows.append((window_start, window_end))

    window_start = window_start + pd.Timedelta(hours=PREDICT_PERIOD + STRIDE)

validation_idx = sorted(set(validation_idx))
dataset_validation = data_2024.loc[validation_idx].copy().sort_values("Time").reset_index(drop=True)
remainder_2024_for_train = data_2024.drop(index=validation_idx).copy().sort_values("Time").reset_index(drop=True)

# Final training set: fixed historical TRAIN_WINDOW + non-validation remainder of 2024.
dataset_train = (
    pd.concat([history, remainder_2024_for_train], ignore_index=True)
    .sort_values("Time")
    .reset_index(drop=True)
)

# Keep 2025 as test set.
dataset_test = dataset_test.copy().reset_index(drop=True)

target_time = val_start_ts
prices = history["DKPrice"].astype(float).values.reshape(-1, 1)

def _load_feature_predictions_for_zone(zone):
    prediction_path = Path(project_root) / "Data" / f"feature_predictions_{zone}_2024-2025.csv"
    if not prediction_path.exists():
        print("No precomputed forecasts found.")
        return None

    predictions = pd.read_csv(prediction_path, decimal=",", sep = ";", parse_dates=["Time"])
    predictions = predictions.loc[:, ~predictions.columns.duplicated()].copy()
    if "DKZone" in predictions.columns:
        predictions = predictions.loc[predictions["DKZone"] == zone].copy()
        print(f"Loaded {len(predictions)} forecasts for zone {zone}.")
        print(f"Forecast features: {len(predictions.columns)} {predictions.columns.tolist()}")
    return predictions

print(f"Using zone: {PRICE_ZONE}")
print(f"Train source shape (all of 2024): {DK1_train.shape if PRICE_ZONE == 'DK1' else DK2_train.shape}")
print(f"Test source shape (all of 2025): {DK1_test.shape if PRICE_ZONE == 'DK1' else DK2_test.shape}")
print(f"Train shape (history + 2024 remainder): {dataset_train.shape}")
print(f"Validation shape (rolling 2024 windows): {dataset_validation.shape}")
print(f"Test shape (2025): {dataset_test.shape}")
print(f"Validation windows created: {len(validation_windows)}")
if validation_windows:
    print("All validation windows:")
    for idx, (window_start, window_end) in enumerate(validation_windows, start=1):
        print(f"  {idx:02d}. {window_start} -> {window_end}")
print(f"Features loaded: {len(dataset_train.columns)} {[c for c in dataset_train.columns if c != 'Time']}")

feature_predictions = _load_feature_predictions_for_zone(PRICE_ZONE)
if feature_predictions is not None:
    print("\nPrecomputed forecasts loaded.")

use_precomputed_feature_values = feature_predictions is not None

Notebook_dir: c:\Users\chris\Documents\Python\Speciale_Kode\Modules
Python_dir: c:\Users\chris\Documents\Python\Speciale_Kode
Data_folder: c:\Users\chris\Documents\Python\Speciale_Kode\Data
Training data shape (DK1): (78888, 38)
Test data shape (DK1): (8760, 38)
Test set fraction (DK1): 9.99%
Training data shape (DK2): (78888, 38)
Test data shape (DK2): (8760, 38)
Test set fraction (DK2): 9.99%
Using zone: DK1
Train source shape (all of 2024): (78888, 38)
Test source shape (all of 2025): (8760, 38)
Train shape (history + 2024 remainder): (23616, 38)
Validation shape (rolling 2024 windows): (2688, 38)
Test shape (2025): (8760, 38)
Validation windows created: 4
All validation windows:
  01. 2024-01-01 00:00:00 -> 2024-01-29 00:00:00
  02. 2024-04-01 00:00:00 -> 2024-04-29 00:00:00
  03. 2024-07-01 00:00:00 -> 2024-07-29 00:00:00
  04. 2024-09-30 00:00:00 -> 2024-10-28 00:00:00
Features loaded: 38 ['DKPrice', 'OffshoreWindPower', 'OnshoreWindPower', 'HydroPower', 'SolarPower', 'Biomass', 

Load Random Forest forecasting models

In [3]:
from Modules.Load_RF_forecast_models import load_rf_models

rf_models = None
if not use_precomputed_feature_values:
    # load_rf_models currently supports only the optional timeout argument.
    rf_models = load_rf_models(user="Christine - desktop")      # set user to "Nikolaj" or "Christine"

Test CUDA

In [4]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA DIAGNOSTICS")
print("\nBasic Info:")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"\nGPU Info:")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"cuDNN Version: {torch.backends.cudnn.version()}")
    print(f"Device Count: {torch.cuda.device_count()}")

    test_tensor = torch.randn(100, 100).to(device)
    print(f"Tensor on CUDA: {test_tensor.is_cuda}")

else:
    print("\n  Running on CPU - no CUDA available")

CUDA DIAGNOSTICS

Basic Info:
CUDA available: True
Device: cuda

GPU Info:
GPU Name: NVIDIA GeForce GTX 1660
CUDA Version: 12.1
cuDNN Version: 90100
Device Count: 1
Tensor on CUDA: True


### Helper functions

In [5]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.base import BaseEstimator, RegressorMixin
from torch.utils.data import DataLoader, TensorDataset

def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def smape_mean(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    denom = np.abs(y_true) + np.abs(y_pred)
    vals = np.where(denom == 0, 0.0, 200.0 * np.abs(y_pred - y_true) / denom)
    return float(np.mean(vals))

class TabularGRU(nn.Module):
    """GRU over true temporal windows: (batch, sequence_length, n_features)."""

    def __init__(self, input_size: int, hidden_size: int, layers: int, dropout: float = 0.0):
        super().__init__()
        self.dropout = float(dropout)
        self.gru = nn.GRU(           
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=layers,
            dropout=self.dropout if layers > 1 else 0.0,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.gru(x)          
        return self.fc(out[:, -1, :])

class TorchGRURegressor(BaseEstimator, RegressorMixin):
    """Scikit-learn style regressor using configurable rolling time sequences."""

    def __init__(
        self,
        hidden_size: int = 32,
        layers: int = 1,
        learning_rate: float = 1e-3,
        epochs: int = 40,
        batch_size: int = 64,
        sequence_length: int = 24,
        dropout: float = 0.0,
        random_state: int = 42,
        log_epoch_metrics: bool = False,
        log_prefix: str = "",
        warm_start: bool = False,
    ):
        self.hidden_size = hidden_size
        self.layers = layers
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.sequence_length = sequence_length
        self.dropout = dropout
        self.random_state = random_state
        self.log_epoch_metrics = log_epoch_metrics
        self.log_prefix = log_prefix
        self.warm_start = warm_start

    def _to_tensor_sequence(self, X):
        X_np = np.asarray(X, dtype=np.float32)
        if X_np.ndim != 2:
            raise ValueError(f"Expected X with shape (n_samples, n_features), got {X_np.shape}.")

        n_samples, n_features = X_np.shape
        seq_len = max(1, int(self.sequence_length))

        if n_samples == 0:
            raise ValueError("X is empty; cannot build sequences.")

        # Left-pad with the first row so each timestamp gets a full sequence window.
        pad = np.repeat(X_np[:1], repeats=seq_len - 1, axis=0)
        padded = np.vstack([pad, X_np])

        X_seq = np.empty((n_samples, seq_len, n_features), dtype=np.float32)
        for i in range(n_samples):
            X_seq[i] = padded[i : i + seq_len]

        return torch.tensor(X_seq, dtype=torch.float32)

    def _initialize_model_state(self, input_size: int):
        self.input_size_ = int(input_size)
        self.device_ = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model_ = TabularGRU(
            input_size=self.input_size_,
            hidden_size=int(self.hidden_size),
            layers=int(self.layers),
            dropout=float(self.dropout),
        ).to(self.device_)
        self.loss_fn_ = nn.MSELoss()
        self.optimizer_ = torch.optim.Adam(self.model_.parameters(), lr=float(self.learning_rate))
        self.epoch_losses_ = []
        self.epoch_smapes_ = []
        self.epoch_maes_ = []
        self.epoch_rmses_ = []
        self._epochs_trained_ = 0

    def fit(self, X, y):
        set_seed(self.random_state)

        X_tensor = self._to_tensor_sequence(X)
        y_np = np.asarray(y, dtype=np.float32).reshape(-1, 1)
        if len(y_np) != len(X_tensor):
            raise ValueError("X and y must have the same number of rows.")
        y_tensor = torch.tensor(y_np, dtype=torch.float32)

        needs_reinit = (
            (not bool(self.warm_start))
            or (not hasattr(self, "model_"))
            or (not hasattr(self, "input_size_"))
            or (int(self.input_size_) != int(X_tensor.shape[-1]))
        )
        if needs_reinit:
            self._initialize_model_state(input_size=int(X_tensor.shape[-1]))

        dataset_local = TensorDataset(X_tensor, y_tensor)
        loader = DataLoader(dataset_local, batch_size=int(self.batch_size), shuffle=True)

        self.model_.train()
        for _ in range(int(self.epochs)):
            batch_losses = []
            epoch_preds = []
            epoch_targets = []

            for X_batch, y_batch in loader:
                X_batch = X_batch.to(self.device_)
                y_batch = y_batch.to(self.device_)
                self.optimizer_.zero_grad()
                preds = self.model_(X_batch)
                loss = self.loss_fn_(preds, y_batch)
                loss.backward()
                self.optimizer_.step()

                batch_losses.append(float(loss.item()))
                epoch_preds.append(preds.detach().cpu().numpy().reshape(-1))
                epoch_targets.append(y_batch.detach().cpu().numpy().reshape(-1))

            epoch_loss = float(np.mean(batch_losses)) if batch_losses else float("nan")
            self.epoch_losses_.append(epoch_loss)

            if epoch_preds and epoch_targets:
                y_pred_epoch = np.concatenate(epoch_preds)
                y_true_epoch = np.concatenate(epoch_targets)
                epoch_smape = smape_mean(y_true_epoch, y_pred_epoch)
                epoch_mae = float(np.mean(np.abs(y_true_epoch - y_pred_epoch)))
                epoch_rmse = float(np.sqrt(np.mean((y_true_epoch - y_pred_epoch) ** 2)))
            else:
                epoch_smape = float("nan")
                epoch_mae = float("nan")
                epoch_rmse = float("nan")

            self.epoch_smapes_.append(float(epoch_smape))
            self.epoch_maes_.append(float(epoch_mae))
            self.epoch_rmses_.append(float(epoch_rmse))

            self._epochs_trained_ += 1

            if bool(self.log_epoch_metrics):
                try:
                    import wandb
                    if wandb.run is not None:
                        loss_name = f"{self.log_prefix}train_MSE_loss" if self.log_prefix else "train_MSE_loss"
                        smape_name = f"{self.log_prefix}train_smape" if self.log_prefix else "train_smape"
                        mae_name = f"{self.log_prefix}train_mae" if self.log_prefix else "train_mae"
                        rmse_name = f"{self.log_prefix}train_rmse" if self.log_prefix else "train_rmse"
                        epoch_name = f"{self.log_prefix}epoch" if self.log_prefix else "epoch"
                        wandb.log({
                            loss_name: epoch_loss,
                            smape_name: float(epoch_smape),
                            mae_name: float(epoch_mae),
                            rmse_name: float(epoch_rmse),
                            epoch_name: int(self._epochs_trained_),
                        })
                except Exception:
                    pass

        return self

    def predict(self, X):
        X_np = np.asarray(X, dtype=np.float32)
        if X_np.ndim == 2:
            X_tensor = self._to_tensor_sequence(X_np)
        elif X_np.ndim == 3:
            X_tensor = torch.tensor(X_np, dtype=torch.float32)
        else:
            raise ValueError(f"Expected X with shape (n_samples, n_features) or (n_samples, sequence_length, n_features), got {X_np.shape}.")
        self.model_.eval()
        with torch.no_grad():
            preds = self.model_(X_tensor.to(self.device_)).squeeze(-1).cpu().numpy()
        return preds

## Hyperparameter search

### Search

Search grid

In [6]:
import numpy as np

param_grid = {
    "hidden_size": [32, 64, 128],
    "layers": [1, 2],
    "learning_rate": [0.001, 0.0005],
    "max_epochs": [100],
    "patience": [6],
    "batch_size": [32, 64],
    "sequence_length": [24, 168],
    "dropout": [0.0, 0.2]
}

print("Total combinations:", np.prod([len(v) for v in param_grid.values()]))

Total combinations: 96


Hyperparameter search

In [ ]:
# Christine API key: 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

In [8]:
# from Modules.Cross_Validation_runner import run_cross_validation
from Modules.Validation3 import run_cross_validation
import itertools
from pathlib import Path
from time import time

import pandas as pd
import wandb


split_setup = 2
train_window = 2 * 8760
val_window = 1 * 8784
val_start = "2024-01-01 00:00:00"
predict_period = 1 * 168  # one 168-hour block per validation fold
stride = 13 * 168

WANDB_PROJECT = "GRU_hyperparameter_search_DK1"
WANDB_RUN_BASENAME = f"{PRICE_ZONE}_gru_hyperparameter_search"

num_combinations = np.prod([len(v) for v in param_grid.values()])
print(f"\nTotal number of combinations to test: {num_combinations}")

param_names = list(param_grid.keys())
param_values = list(param_grid.values())
all_combinations = list(itertools.product(*param_values))

start_time = time()
results = []
for comb_number, combination in enumerate(all_combinations, start=1):
    params = dict(zip(param_names, combination))
    if comb_number <= 8:     # Change 0 to the number of the last completed combination.
        continue
    if int(params["layers"]) == 1 and float(params["dropout"]) > 0.0:
        continue

    print(f"\nCombination {comb_number}/{num_combinations}: {params}")
    print(
        f"Time: {(time() - start_time)/60:.2f} minutes - estimated total time: "
        f"{(time() - start_time)/comb_number*num_combinations/60:.2f} minutes"
    )

    run_name = f"{WANDB_RUN_BASENAME}_comb_{comb_number:03d}"
    run = wandb.init(
        project=WANDB_PROJECT,
        name=run_name,
        config={
            "price_zone": PRICE_ZONE,
            "train_window": train_window,
            "val_window": val_window,
            "val_start": val_start,
            "predict_period": predict_period,
            "stride": stride,
            "split_setup": split_setup,
            "combination": int(comb_number),
            "num_combinations": int(num_combinations),
            **params,
        },
        tags=["gru", "hyperparameter-search", "cross-validation", "early-stopping"],
        reinit=True,
#        settings=wandb.Settings(start_method="fork"),
    )

    try:
        max_epochs = int(params["max_epochs"])
        patience = int(params["patience"])

        model = TorchGRURegressor(
            hidden_size=int(params["hidden_size"]),
            layers=int(params["layers"]),
            learning_rate=float(params["learning_rate"]),
            epochs=1,
            batch_size=int(params["batch_size"]),
            sequence_length=int(params["sequence_length"]),
            dropout=float(params["dropout"]),
            random_state=42,
            warm_start=True,
        )

        best_val_smape = float("inf")
        best_epoch = 0
        patience_counter = 0
        best_combination_results = None

        for epoch in range(1, max_epochs + 1):
            print(f"  Epoch {epoch}/{max_epochs}")
            combination_results = run_cross_validation(
                model=model,
                dataset_train=dataset_train,
                dataset_validation=dataset_validation,
                include_remaining_2024=True,
                dk_zone=PRICE_ZONE,
                split_setup=split_setup,
                train_window=train_window,
                val_window=val_window,
                val_start=val_start,
                predict_period=predict_period,
                stride=stride,
                use_scaler=True,
                print_fold_results=False,
                plot=False,
                rf_models=rf_models,
                use_precomputed_feature_values=use_precomputed_feature_values,
                precomputed_feature_predictions=feature_predictions,
                use_forecasted_history=True,
            )

            val_smape = float(combination_results["overall_avg_weekly_smape"])

            if val_smape < best_val_smape:
                best_val_smape = val_smape
                best_epoch = epoch
                best_combination_results = combination_results
                patience_counter = 0
            else:
                patience_counter += 1

            wandb.log({
                "combination": int(comb_number),
                "epoch": int(epoch),
                "train_window": int(train_window),
                "val_window": int(val_window),
                "predict_period": int(predict_period),
                "hidden_size": int(params["hidden_size"]),
                "layers": int(params["layers"]),
                "learning_rate": float(params["learning_rate"]),
                "batch_size": int(params["batch_size"]),
                "sequence_length": int(params["sequence_length"]),
                "max_epochs": int(params["max_epochs"]),
                "patience": int(params["patience"]),
                "val_SMAPE": float(val_smape),
                "best_val_SMAPE": float(best_val_smape),
                "patience_counter": int(patience_counter),
            })

            if patience_counter >= patience:
                print("  Early stopping triggered.")
                break

        print(f"\n  best_val_SMAPE={best_val_smape:.3f}")

        if best_combination_results is None:
            raise RuntimeError("No validation results were produced for this combination.")

        row = {
            **params,
            "best_epoch": int(best_epoch),
            "epochs_trained": int(epoch),
            "price_zone": PRICE_ZONE,
            "train_window": str(train_window // 8760) + " years",
            "val_start": val_start.split(" ")[0],
            "avg_smape": best_val_smape,
            "avg_weekly_rmse": best_combination_results["overall_avg_weekly_rmse"],
            "avg_weekly_mae": best_combination_results["overall_avg_weekly_mae"],
            "avg_weekly_smape": best_combination_results["overall_avg_weekly_smape"],
            "avg_daily_rmse": best_combination_results["overall_avg_daily_rmse"],
            "avg_daily_mae": best_combination_results["overall_avg_daily_mae"],
            "avg_daily_smape": best_combination_results["overall_avg_daily_smape"],
            "avg_smape_day_1": best_combination_results["avg_smape_day_1"],
            "avg_smape_day_2": best_combination_results["avg_smape_day_2"],
            "avg_smape_day_3": best_combination_results["avg_smape_day_3"],
            "avg_smape_day_4": best_combination_results["avg_smape_day_4"],
            "avg_smape_day_5": best_combination_results["avg_smape_day_5"],
            "avg_smape_day_6": best_combination_results["avg_smape_day_6"],
            "avg_smape_day_7": best_combination_results["avg_smape_day_7"],
        }
        results.append(row)

        run.summary.update({
            "best_epoch": int(best_epoch),
            "best_val_smape": float(best_val_smape),
            "epochs_trained": int(epoch),
        })
    finally:
        wandb.finish()

results_df = pd.DataFrame(results).sort_values("avg_smape")

project_root = Path.cwd()
while project_root.name != "Speciale_Kode" and project_root.parent != project_root:
    project_root = project_root.parent

output_folder = project_root / "Deep learners" / "GRU"
output_folder.mkdir(parents=True, exist_ok=True)

base_filename = f"{PRICE_ZONE}_gru_multi_search_results"
filename = output_folder / f"{base_filename}.csv"
counter = 1
while filename.exists():
    filename = output_folder / f"{base_filename}_{counter}.csv"
    counter += 1

results_df.to_csv(filename, index=False, decimal=",")
print(f"\nResults saved to: {filename}")
display(results_df.head(10))


Total number of combinations to test: 96



Combination 9/96: {'hidden_size': 32, 'layers': 1, 'learning_rate': 0.0005, 'max_epochs': 100, 'patience': 6, 'batch_size': 32, 'sequence_length': 24, 'dropout': 0.0}
Time: 0.00 minutes - estimated total time: 0.00 minutes


  Epoch 1/100
Model trained in 2.12s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 178.757
  Epoch 2/100
Model trained in 1.86s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 169.093
  Epoch 3/100
Model trained in 1.89s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 160.419
  Epoch 4/100
Model trained in 1.89s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 153.180
  Epoch 5/100
Model trained in 2.22s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 146.779
  Epoch 6/100
Model trained in 1.87s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 140.893
  Epoch 7/100
Model trained in 1.93s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 135.489
  Epoch 8/100
Model trained in 1.86s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 130.402
  Epoch 9/100
Model trained in 2

batch_size,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val_SMAPE,█▇▇▆▆▆▅▅▅▄▄▄▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
combination,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
hidden_size,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
layers,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
max_epochs,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
patience,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
patience_counter,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▂▃▅▆▇█
+5,...



Combination 11/96: {'hidden_size': 32, 'layers': 1, 'learning_rate': 0.0005, 'max_epochs': 100, 'patience': 6, 'batch_size': 32, 'sequence_length': 168, 'dropout': 0.0}
Time: 43.40 minutes - estimated total time: 378.75 minutes


...


Combination 96/96: {'hidden_size': 128, 'layers': 2, 'learning_rate': 0.0005, 'max_epochs': 100, 'patience': 6, 'batch_size': 64, 'sequence_length': 168, 'dropout': 0.2}
Time: 2639.32 minutes - estimated total time: 2639.32 minutes


  Epoch 1/100
Model trained in 4.00s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 167.298
  Epoch 2/100
Model trained in 3.70s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 151.912
  Epoch 3/100
Model trained in 3.78s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 139.350
  Epoch 4/100
Model trained in 3.63s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 129.926
  Epoch 5/100
Model trained in 3.68s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 122.415
  Epoch 6/100
Model trained in 3.69s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 114.019
  Epoch 7/100
Model trained in 3.69s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 106.885
  Epoch 8/100
Model trained in 4.67s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 100.254
  Epoch 9/100
Model trained in 3

batch_size,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val_SMAPE,█▇▆▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
combination,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇██
hidden_size,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
layers,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
max_epochs,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
patience,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
patience_counter,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▃▅▆▇█
+5,...



Results saved to: c:\Users\chris\Documents\Python\Speciale_Kode\Deep learners\GRU\DK1_gru_multi_search_results_3.csv


,hidden_size,layers,learning_rate,max_epochs,patience,batch_size,sequence_length,dropout,best_epoch,epochs_trained,...,avg_daily_rmse,avg_daily_mae,avg_daily_smape,avg_smape_day_1,avg_smape_day_2,avg_smape_day_3,avg_smape_day_4,avg_smape_day_5,avg_smape_day_6,avg_smape_day_7
51,128,1,0.0005,100,6,64,168,0.0,23,29,...,255.635253,220.544410,61.980932,64.453786,52.729567,37.745482,52.359096,56.418158,77.294541,92.865894
49,128,1,0.0005,100,6,32,168,0.0,11,17,...,249.506333,214.725809,62.031924,61.728004,49.922916,38.721423,53.037297,58.358114,78.475291,93.980425
39,64,2,0.0005,100,6,32,168,0.2,21,27,...,256.018842,220.727293,62.067218,64.277375,52.685489,38.253799,52.438567,56.517724,77.383783,92.913785
17,32,2,0.0005,100,6,64,24,0.2,81,87,...,256.164400,220.862043,62.067429,64.340723,52.742694,38.224815,52.435152,56.475800,77.360453,92.892365
56,128,2,0.0010,100,6,64,24,0.0,11,17,...,255.448939,220.205066,62.069917,64.033806,52.454372,38.370336,52.467232,56.686327,77.477464,92.999883
59,128,2,0.0010,100,6,64,168,0.2,11,17,...,255.406256,220.165697,62.070231,64.015053,52.436576,38.379271,52.470334,56.699258,77.484639,93.006483
58,128,2,0.0010,100,6,64,168,0.0,11,17,...,255.404277,220.163868,62.070245,64.014181,52.435750,38.379686,52.470478,56.699858,77.484972,93.006790
43,64,2,0.0005,100,6,64,168,0.2,42,48,...,256.860339,221.518024,62.073945,64.638662,53.019014,38.115171,52.418851,56.281133,77.251936,92.792848
4,32,2,0.0010,100,6,32,24,0.0,21,27,...,257.055799,221.711317,62.078486,64.722726,53.094945,38.095494,52.416238,56.228033,77.226264,92.765702
5,32,2,0.0010,100,6,32,24,0.2,21,27,...,257.056016,221.711536,62.078492,64.722800,53.095012,38.095477,52.416237,56.228004,77.226243,92.765670


## Find best number of epochs

In [ ]:
import numpy as np
import pandas as pd
import wandb
from Modules.Validation3 import run_cross_validation

# Find best number of epochs with validation windows from dataset_validation
# ======================================================================
PRICE_ZONE = "DK1"
TRAIN_WINDOW = 2 * 8760
VAL_START = "2024-01-01 00:00:00"
VAL_WINDOW = 1 * 8784
PREDICT_PERIOD = 4 * 168
STRIDE = 13 * 168
MAX_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 10
MIN_DELTA = 0.0
INCLUDE_REMAINING_2024_DURING_SEARCH = True
WANDB_PROJECT = "GRU_DK1"
WANDB_RUN_NAME = f"{PRICE_ZONE}_gru_epoch_search"

param_grid = {
    "hidden_size": [128],
    "layers": [1],
    "learning_rate": [0.0005],
    "batch_size": [32],
    "sequence_length": [168],
    "dropout": [0.0]
}

params = {key: values[0] for key, values in param_grid.items()}

if PRICE_ZONE not in ["DK1", "DK2"]:
    raise ValueError("PRICE_ZONE must be 'DK1' or 'DK2'.")
if "dataset_train" not in globals() or "dataset_validation" not in globals():
    raise ValueError("Run Cell 6 first to create dataset_train and dataset_validation.")

train_data = dataset_train.copy()
validation_data = dataset_validation.copy()

if validation_data.empty:
    raise ValueError("dataset_validation is empty; cannot run epoch search with validation.")

run = wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    config={
        "price_zone": PRICE_ZONE,
        "train_window": TRAIN_WINDOW,
        "val_start": VAL_START,
        "val_window": VAL_WINDOW,
        "predict_period": PREDICT_PERIOD,
        "stride": STRIDE,
        "max_epochs": MAX_EPOCHS,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "early_stopping_min_delta": MIN_DELTA,
        "include_remaining_2024_during_search": INCLUDE_REMAINING_2024_DURING_SEARCH,
        **params,
    },
    tags=["gru", "epoch-search", "cross-validation", "early-stopping"],
    reinit=True,
    settings=wandb.Settings(start_method="thread"),
)

print(f"Epoch search train source shape: {train_data.shape}")
print(f"Epoch search validation source shape: {validation_data.shape}")

# Warm-start model: 1 epoch per call so we can evaluate every epoch on validation
model = TorchGRURegressor(
    hidden_size=int(params["hidden_size"]),
    layers=int(params["layers"]),
    learning_rate=float(params["learning_rate"]),
    epochs=1,
    batch_size=int(params["batch_size"]),
    sequence_length=int(params["sequence_length"]),
    random_state=42,
    log_epoch_metrics=False,
    log_prefix="",
    warm_start=True,
)

epoch_history = []
best_val_smape = float("inf")
best_epoch = 0
patience_counter = 0

for epoch in range(1, MAX_EPOCHS + 1):
    print(f"\n=== Epoch {epoch}/{MAX_EPOCHS} ===")
    combination_results = run_cross_validation(
        model=model,
        dataset_train=train_data,
        dataset_validation=validation_data,
        include_remaining_2024=INCLUDE_REMAINING_2024_DURING_SEARCH,
        dk_zone=PRICE_ZONE,
        split_setup=2,
        train_window=TRAIN_WINDOW,
        val_window=VAL_WINDOW,
        val_start=VAL_START,
        predict_period=PREDICT_PERIOD,
        stride=STRIDE,
        use_scaler=True,
        print_fold_results=False,
        plot=False,
        rf_models=rf_models,
        use_precomputed_feature_values=use_precomputed_feature_values,
        precomputed_feature_predictions=feature_predictions,
        use_forecasted_history=True,
    )

    trained_model = combination_results.get("model", model)

    train_mse_loss = float(trained_model.epoch_losses_[-1]) if hasattr(trained_model, "epoch_losses_") else float("nan")
    train_smape = float(trained_model.epoch_smapes_[-1]) if hasattr(trained_model, "epoch_smapes_") else float("nan")
    train_mae = float(trained_model.epoch_maes_[-1]) if hasattr(trained_model, "epoch_maes_") else float("nan")
    train_rmse = float(trained_model.epoch_rmses_[-1]) if hasattr(trained_model, "epoch_rmses_") else float("nan")

    val_smape = float(combination_results["overall_avg_weekly_smape"])
    val_mae = float(combination_results["overall_avg_weekly_mae"])
    val_rmse = float(combination_results["overall_avg_weekly_rmse"])
    val_mse_loss = float(val_rmse ** 2)

    epoch_row = {
        "epoch": epoch,
        "train_MSE_loss": train_mse_loss,
        "train_SMAPE": train_smape,
        "train_MAE": train_mae,
        "train_RMSE": train_rmse,
        "val_MSE_loss": val_mse_loss,
        "val_SMAPE": val_smape,
        "val_MAE": val_mae,
        "val_RMSE": val_rmse,
    }
    epoch_history.append(epoch_row)
    wandb.log(epoch_row)

    improved = val_smape < (best_val_smape - MIN_DELTA)
    if improved:
        best_val_smape = val_smape
        best_epoch = epoch
        patience_counter = 0
    else:
        patience_counter += 1

    print(
        f"\nEpoch {epoch:03d} | "
        f"train_SMAPE={train_smape:.3f} | val_SMAPE={val_smape:.3f} | "
        f"best_val_SMAPE={best_val_smape:.3f} | "
        f"patience={patience_counter}/{EARLY_STOPPING_PATIENCE}"
    )

    if patience_counter >= EARLY_STOPPING_PATIENCE:
        print(f"Early stopping triggered at epoch {epoch}. Best epoch: {best_epoch}.")
        break

if best_epoch == 0:
    raise RuntimeError("No valid epoch found during validation-based epoch search.")

epoch_metrics_df = pd.DataFrame(epoch_history)
wandb.log({"epoch_search_metrics": wandb.Table(dataframe=epoch_metrics_df)})
wandb.log({
    "best_epoch": int(best_epoch),
    "best_val_smape": float(best_val_smape),
})

run.summary.update({
    "best_epoch": int(best_epoch),
    "best_val_smape": float(best_val_smape),
})

# Expose best epoch for the next cell
BEST_EPOCHS = int(best_epoch)
BEST_EPOCH_SEARCH_HISTORY = epoch_metrics_df.copy()

print(f"\nSelected BEST_EPOCHS = {BEST_EPOCHS}")

wandb.finish()

## Train final model

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import wandb
import tempfile
import joblib

# Train final model on the last TRAIN_HOURS of 2024 with BEST_EPOCHS (no epoch validation)
# =====================================================================================
PRICE_ZONE = "DK1"
TRAIN_HOURS = 2 * 8760
WANDB_PROJECT = "GRU_DK1"
WANDB_RUN_NAME = f"{PRICE_ZONE}_gru_multivariate_final"
save_model_to_disk = False

if "BEST_EPOCHS" not in globals():
    raise ValueError("Run the previous epoch-search cell first to define BEST_EPOCHS.")

if PRICE_ZONE == "DK1":
    train_source = DK1_train.copy()
elif PRICE_ZONE == "DK2":
    train_source = DK2_train.copy()
else:
    raise ValueError("PRICE_ZONE must be 'DK1' or 'DK2'.")

train_source = train_source.sort_values("Time").reset_index(drop=True)
if len(train_source) < TRAIN_HOURS:
    raise ValueError(
        f"TRAIN_HOURS={TRAIN_HOURS} exceeds available rows ({len(train_source)}) in {PRICE_ZONE} train set."
    )

train_set = train_source.tail(int(TRAIN_HOURS)).copy().reset_index(drop=True)
feature_columns = [c for c in train_set.columns if c not in ["Time", "DKPrice"]]
X_full = train_set[feature_columns]
y_full = train_set["DKPrice"]

param_grid = {
    "hidden_size": [128],
    "layers": [1],
    "learning_rate": [0.0005],
    "batch_size": [32],
    "sequence_length": [168],
    "dropout": [0.0]
}

output_root = Path(project_root) / "Deep learners" / "GRU"
output_root.mkdir(parents=True, exist_ok=True)

run = wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    config={
        "price_zone": PRICE_ZONE,
        "training_period": "tail_train_hours",
        "train_hours": int(TRAIN_HOURS),
        "training_rows": int(len(train_set)),
        "train_start_time": str(train_set["Time"].min()),
        "train_end_time": str(train_set["Time"].max()),
        "epochs": int(BEST_EPOCHS),
        **params,
    },
    tags=["gru", "final-model", "tail-train-hours", "no-epoch-validation"],
    reinit=True,
    settings=wandb.Settings(start_method="thread"),
)

print(f"Final training rows: {len(train_set)}")
print(f"Final training window: {train_set['Time'].min()} -> {train_set['Time'].max()}")

final_model = TorchGRURegressor(
    hidden_size=int(params["hidden_size"]),
    layers=int(params["layers"]),
    learning_rate=float(params["learning_rate"]),
    epochs=int(BEST_EPOCHS),
    batch_size=int(params["batch_size"]),
    sequence_length=int(params["sequence_length"]),
    random_state=42,
    log_epoch_metrics=True,
    log_prefix="final_tailhours_",
    warm_start=False,
)

final_model.fit(X_full, y_full)
model = final_model

# Save per-epoch training metrics from final training
if hasattr(final_model, "epoch_losses_") and len(final_model.epoch_losses_) > 0:
    final_train_metrics_df = pd.DataFrame({
        "epoch": np.arange(1, len(final_model.epoch_losses_) + 1),
        "train_MSE_loss": final_model.epoch_losses_,
        "train_SMAPE": final_model.epoch_smapes_,
        "train_MAE": final_model.epoch_maes_,
        "train_RMSE": final_model.epoch_rmses_,
    })
    wandb.log({"final_tailhours_epoch_metrics": wandb.Table(dataframe=final_train_metrics_df)})
    wandb.log({
        "final_tailhours_last_epoch_train_MSE_loss": float(final_model.epoch_losses_[-1]),
        "final_tailhours_last_epoch_train_SMAPE": float(final_model.epoch_smapes_[-1]),
        "final_tailhours_last_epoch_train_MAE": float(final_model.epoch_maes_[-1]),
        "final_tailhours_last_epoch_train_RMSE": float(final_model.epoch_rmses_[-1]),
    })

# Save model artifact to W&B
model_artifact = wandb.Artifact(name=f"{WANDB_RUN_NAME}_model", type="model")
with tempfile.TemporaryDirectory() as tmpdir:
    model_path = Path(tmpdir) / f"{WANDB_RUN_NAME}_model.joblib"
    joblib.dump(final_model, model_path, compress=3)
    model_artifact.add_file(str(model_path), name="model.joblib")
    run.log_artifact(model_artifact)

print(f"Trained final model on last TRAIN_HOURS={TRAIN_HOURS} of {PRICE_ZONE} train set with BEST_EPOCHS={BEST_EPOCHS}.")
print("Model stored in W&B artifact.")

if save_model_to_disk:
    print("Local model persistence is disabled; model was stored in W&B instead.")

wandb.finish()

Shap analysis

In [ ]:
import gc
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import shap
import wandb

# ==========================
# SHAP configuration (GRU)
# ==========================
quick_mode = False
eval_size = 300            # evaluation sample size
bg_size = 150              # background sample size for KernelExplainer
chunk_size = 25            # lower if memory/runtime is high
nsamples = 100             # SHAP Monte Carlo samples per explained point
include_beeswarm = True
include_waterfall = True

PRICE_ZONE = globals().get("PRICE_ZONE", "DK1")
TRAIN_HOURS = int(globals().get("TRAIN_HOURS", 2 * 8760))
WANDB_PROJECT = globals().get("WANDB_PROJECT", "GRU_DK1")
WANDB_RUN_NAME = globals().get("WANDB_RUN_NAME", f"{PRICE_ZONE}_gru_multivariate_final")
WANDB_ARTIFACT_NAME = f"{WANDB_RUN_NAME}_model"
WANDB_SHAP_RUN_NAME = f"{PRICE_ZONE}_gru_shap"

if PRICE_ZONE == "DK1":
    train_source = DK1_train.copy() if "DK1_train" in globals() else None
elif PRICE_ZONE == "DK2":
    train_source = DK2_train.copy() if "DK2_train" in globals() else None
else:
    raise ValueError("PRICE_ZONE must be 'DK1' or 'DK2'.")

if train_source is None:
    raise ValueError("Missing train source data. Run the data loading cell first.")

train_source = train_source.sort_values("Time").reset_index(drop=True)
if len(train_source) < TRAIN_HOURS:
    raise ValueError(
        f"TRAIN_HOURS={TRAIN_HOURS} exceeds available rows ({len(train_source)}) in {PRICE_ZONE} train set."
    )

# Use the same final-training window definition: tail(TRAIN_HOURS)
train_frame = train_source.tail(TRAIN_HOURS).copy().reset_index(drop=True)
feature_columns = [c for c in train_frame.columns if c not in ["Time", "DKPrice"]]
if not feature_columns:
    raise ValueError("No feature columns found after removing ['Time', 'DKPrice'].")

X_train_shap = train_frame.loc[:, feature_columns].copy()
if len(X_train_shap) == 0:
    raise ValueError("No rows found in TRAIN_HOURS tail slice for SHAP analysis.")

X_train_shap = X_train_shap.astype(np.float32, copy=False)

# Start a dedicated W&B run for SHAP and load latest model artifact
wandb_run = wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_SHAP_RUN_NAME,
    job_type="shap-analysis",
    config={
        "price_zone": PRICE_ZONE,
        "train_hours": int(TRAIN_HOURS),
        "artifact_name": WANDB_ARTIFACT_NAME,
        "eval_size_requested": int(eval_size),
        "bg_size_requested": int(bg_size),
        "nsamples": int(nsamples),
    },
    reinit=True,
    settings=wandb.Settings(start_method="thread"),
)

model_artifact = wandb_run.use_artifact(f"{WANDB_ARTIFACT_NAME}:latest")
artifact_dir = Path(model_artifact.download())
model_path = artifact_dir / "model.joblib"
if not model_path.exists():
    raise ValueError(f"Could not find model.joblib in downloaded artifact: {artifact_dir}")

model = joblib.load(model_path)
print(f"Loaded GRU model artifact: {WANDB_ARTIFACT_NAME}:latest")

mem = psutil.virtual_memory()
print(f"Available RAM before SHAP: {mem.available / (1024**3):.2f} GB")
print(f"Training set size for SHAP: {len(X_train_shap)} samples")
print(f"Number of features: {len(feature_columns)}")

if quick_mode:
    bg_size = min(30, len(X_train_shap))
    eval_size = min(60, len(X_train_shap))
    chunk_size = 10
    nsamples = 50
else:
    bg_size = min(bg_size, len(X_train_shap))
    eval_size = min(eval_size, len(X_train_shap))
    chunk_size = max(1, min(chunk_size, eval_size))

X_bg = shap.sample(X_train_shap, bg_size, random_state=42)
X_eval = shap.sample(X_train_shap, eval_size, random_state=42)

print(f"\nBackground sample size (X_bg): {len(X_bg)}")
print(f"Evaluation sample size (X_eval): {len(X_eval)}")
print(f"Chunk size: {chunk_size}")
print(f"Kernel SHAP nsamples: {nsamples}")

def predict_fn(x):
    x_df = pd.DataFrame(x, columns=feature_columns)
    preds = model.predict(x_df)
    return np.asarray(preds).reshape(-1)

explainer = shap.KernelExplainer(predict_fn, X_bg.values)

# Compute SHAP values in chunks and print progress
shap_chunks = []
n_chunks = (len(X_eval) + chunk_size - 1) // chunk_size
for idx, start in enumerate(range(0, len(X_eval), chunk_size), start=1):
    stop = min(start + chunk_size, len(X_eval))
    print(f"Computing SHAP chunk {idx}/{n_chunks} (rows {start}:{stop})...", flush=True)
    X_chunk = X_eval.iloc[start:stop]
    shap_chunk = explainer.shap_values(X_chunk.values, nsamples=nsamples)
    shap_chunks.append(np.asarray(shap_chunk))

shap_values = np.vstack(shap_chunks)
gc.collect()

print("\nSHAP analysis complete.")

# Global importance bar plot
plt.figure(figsize=(12, 6))
shap.summary_plot(shap_values, X_eval, plot_type="bar", show=False)
plt.tight_layout()
wandb_run.log({"shap_bar": wandb.Image(plt.gcf())})
plt.show()
plt.close()

# Beeswarm plot
if include_beeswarm:
    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values, X_eval, show=False)
    plt.tight_layout()
    wandb_run.log({"shap_beeswarm": wandb.Image(plt.gcf())})
    plt.show()
    plt.close()

# Waterfall plot for first sample
if include_waterfall:
    i = 0
    base_value = float(np.mean(predict_fn(X_bg.values)))
    explanation = shap.Explanation(
        values=shap_values[i],
        base_values=base_value,
        data=X_eval.iloc[i].values,
        feature_names=X_eval.columns.tolist(),
    )
    plt.figure(figsize=(10, 4))
    shap.plots.waterfall(explanation, show=False)
    plt.tight_layout()
    wandb_run.log({"shap_waterfall": wandb.Image(plt.gcf())})
    plt.show()
    plt.close()

# Log mean absolute SHAP as a table
mean_abs_shap = np.abs(shap_values).mean(axis=0)
importance_df = pd.DataFrame({
    "feature": feature_columns,
    "mean_abs_shap": mean_abs_shap,
}).sort_values("mean_abs_shap", ascending=False)

wandb_run.log({"shap_importance_table": wandb.Table(dataframe=importance_df)})
wandb_run.summary.update({
    "shap_eval_size": int(len(X_eval)),
    "shap_bg_size": int(len(X_bg)),
    "top_feature": str(importance_df.iloc[0]["feature"]),
    "top_feature_mean_abs_shap": float(importance_df.iloc[0]["mean_abs_shap"]),
})

mem_after = psutil.virtual_memory()
print(f"Available RAM after SHAP cleanup: {mem_after.available / (1024**3):.2f} GB")

# Cleanup large objects explicitly
del X_bg, X_eval, X_train_shap, shap_chunks, shap_values
gc.collect()

wandb.finish()